# 🐼 Pandas — Complete Guide (Basic to Advanced)

This notebook teaches **pandas**, the most popular Python library for working with tabular data (think: Excel sheets, but in code).

**How this notebook is organized:**
1. What is pandas & installation
2. Series — the 1D building block
3. DataFrame — the 2D building block (the real workhorse)
4. Reading & inspecting data
5. Selecting & filtering data
6. Adding, modifying, deleting columns/rows
7. Handling missing data
8. Sorting & ranking
9. GroupBy — aggregation superpower
10. Merging, joining, concatenating
11. Pivot tables & reshaping
12. Apply / map / lambda — custom transformations
13. String operations
14. Date & time handling
15. Advanced: MultiIndex, window functions, performance tips
16. Duplicates
17. set_index() / reset_index()
18. Iterating over rows (and why to avoid it)
19. Reshaping: melt(), stack(), unstack()
20. Categorical data type
21. query() and eval()
22. Exporting data
23. Reading large files in chunks
24. Quick plotting

Every code cell has comments explaining **what** and **why**, not just **how**.
Run cells top to bottom — later cells often reuse variables from earlier ones.

## 1. What is pandas?

**Pandas** = **Pan**el **Da**ta. It gives you two main data structures:

| Structure | Dimensions | Think of it as |
|---|---|---|
| `Series` | 1D | A single column / a labeled list |
| `DataFrame` | 2D | A whole spreadsheet / SQL table |

**Why use pandas instead of plain Python lists/dicts?**
- Fast (built on NumPy, written in C under the hood)
- Handles missing data gracefully
- Reads/writes CSV, Excel, JSON, SQL, etc. in one line
- Powerful filtering, grouping, and reshaping tools

Install (only needed once, outside the notebook, in your terminal):
```
pip install pandas
```

In [ ]:
# The convention EVERYONE uses: import pandas as pd
import pandas as pd
import numpy as np  # pandas leans heavily on numpy under the hood

# Check the version you have installed
print("Pandas version:", pd.__version__)

## 2. Series — the 1D building block

A `Series` is like a Python list, but every value has a **label** (called the *index*).
Think of it as a single column from a spreadsheet, with row labels attached.

In [ ]:
# ─── Creating a Series from a list ───────────────────────────────────────────
# If you don't give an index, pandas auto-assigns 0, 1, 2, ... like a list.
marks = pd.Series([85, 92, 78, 65])
print(marks)
print(type(marks))   # <class 'pandas.core.series.Series'>

In [ ]:
# ─── Creating a Series with a CUSTOM index (labels) ─────────────────────────
# Now each value has a meaningful name instead of just a number.
marks = pd.Series([85, 92, 78, 65], index=["Math", "Science", "English", "Art"])
print(marks)

# Access a value by its LABEL (like a dictionary key)
print("\nMath score:", marks["Math"])

# Access a value by its POSITION (like a list index)
print("First score (position 0):", marks.iloc[0])

In [ ]:
# ─── Creating a Series from a dictionary ─────────────────────────────────────
# Dictionary keys automatically become the index.
population = pd.Series({"India": 1417, "USA": 331, "China": 1412})  # in millions
print(population)

In [ ]:
# ─── Basic Series operations ─────────────────────────────────────────────────
print("Mean:", marks.mean())
print("Max :", marks.max())
print("Min :", marks.min())
print("Sum :", marks.sum())

# Series math is VECTORIZED — this applies to every element at once,
# no manual loop needed (much faster than a Python for-loop).
bonus_marks = marks + 5
print("\nWith +5 bonus applied to everyone:")
print(bonus_marks)

## 3. DataFrame — the 2D building block (the real workhorse)

A `DataFrame` is a table: rows + columns, each column is technically a `Series`.
This is what you'll use 95% of the time in real projects.

In [ ]:
# ─── Creating a DataFrame from a dictionary of lists ────────────────────────
# Each KEY becomes a column name. Each LIST becomes that column's values.
# All lists must be the same length.
data = {
    "Name":  ["Aman", "Priya", "Rohit", "Sneha"],
    "Age":   [23, 25, 22, 24],
    "City":  ["Delhi", "Mumbai", "Pune", "Delhi"],
    "Marks": [85, 92, 78, 88],
}

df = pd.DataFrame(data)
print(df)
print(type(df))   # <class 'pandas.core.frame.DataFrame'>

In [ ]:
# ─── Creating a DataFrame from a list of dictionaries ───────────────────────
# Each dictionary in the list = one ROW. Handy when data comes from an API.
records = [
    {"Name": "Aman", "Age": 23, "City": "Delhi"},
    {"Name": "Priya", "Age": 25, "City": "Mumbai"},
]
df_from_records = pd.DataFrame(records)
print(df_from_records)

## 4. Reading & inspecting data

In real projects you rarely type data by hand — you READ it from a file.
Below we simulate that by saving our `df` to CSV first, then reading it back,
so this notebook works standalone without needing an external file.

In [ ]:
# ─── Save our sample DataFrame to CSV, then read it back ───────────────────
# index=False means: don't write the row numbers (0,1,2...) as a column.
df.to_csv("sample_students.csv", index=False)

# Now read it back — this is exactly how you'd load a real CSV file.
df = pd.read_csv("sample_students.csv")
print(df)

# Other common readers you'll use with real files:
#   pd.read_excel("file.xlsx")
#   pd.read_json("file.json")
#   pd.read_sql(query, connection)

In [ ]:
# ─── Inspecting a DataFrame — the FIRST things to run on any new dataset ────

print("--- head(): first 5 rows (default) ---")
print(df.head())        # df.head(2) → first 2 rows only

print("\n--- tail(): last rows ---")
print(df.tail(2))       # last 2 rows

print("\n--- shape: (rows, columns) ---")
print(df.shape)

print("\n--- columns: list of column names ---")
print(df.columns)

print("\n--- dtypes: data type of each column ---")
print(df.dtypes)

In [ ]:
# ─── info() and describe() — quick health check on your data ───────────────

print("--- info(): column names, non-null counts, dtypes, memory usage ---")
df.info()

print("\n--- describe(): statistical summary of NUMERIC columns ---")
# Shows count, mean, std, min, 25%/50%/75% percentiles, max — all at once!
print(df.describe())

## 5. Selecting & filtering data

This is the single most-used skill in pandas: "give me just the rows/columns I want."

Two key tools:
- `.loc[]`  → select by **label** (column name, index label)
- `.iloc[]` → select by **integer position** (like list indexing)

In [ ]:
# ─── Selecting a single column → returns a Series ───────────────────────────
print(df["Name"])
print(type(df["Name"]))   # Series (single column)

# ─── Selecting multiple columns → returns a DataFrame ───────────────────────
# Note the DOUBLE square brackets: outer [] = "selecting", inner [] = "a list"
print("\n", df[["Name", "Marks"]])

In [ ]:
# ─── .loc[] — select by LABEL ────────────────────────────────────────────────
# Syntax: df.loc[row_labels, column_labels]

print("Row at index label 0:")
print(df.loc[0])

print("\nRows 0-2, only Name and Marks columns:")
print(df.loc[0:2, ["Name", "Marks"]])   # NOTE: .loc slicing is INCLUSIVE of the end

In [ ]:
# ─── .iloc[] — select by INTEGER POSITION ────────────────────────────────────
# Syntax: df.iloc[row_positions, column_positions]  (works just like list/array slicing)

print("First row, by position:")
print(df.iloc[0])

print("\nRows 0-1 (position, EXCLUSIVE of end), columns 0-1:")
print(df.iloc[0:2, 0:2])

In [ ]:
# ─── Boolean / conditional filtering — the "WHERE clause" of pandas ─────────
# df["Marks"] > 80 creates a Series of True/False.
# Passing that Series into df[...] keeps only the True rows.

high_scorers = df[df["Marks"] > 80]
print("Students with Marks > 80:")
print(high_scorers)

# ─── Combining multiple conditions ───────────────────────────────────────────
# Use & (and), | (or), ~ (not) — NOT the Python keywords 'and'/'or'.
# Always wrap each condition in parentheses!
delhi_high_scorers = df[(df["City"] == "Delhi") & (df["Marks"] > 80)]
print("\nDelhi students with Marks > 80:")
print(delhi_high_scorers)

## 6. Adding, modifying, and deleting columns/rows

In [ ]:
# ─── Adding a new column ─────────────────────────────────────────────────────
# Just assign to a column name that doesn't exist yet — pandas creates it.
df["Passed"] = df["Marks"] >= 80          # boolean column, vectorized comparison
df["Grade_Bonus"] = df["Marks"] + 5       # numeric column, vectorized math
print(df)

In [ ]:
# ─── Modifying values ────────────────────────────────────────────────────────
# Use .loc[row, column] to safely modify a specific cell (avoids warnings).
df.loc[0, "Marks"] = 90
print(df)

In [ ]:
# ─── Deleting a column ───────────────────────────────────────────────────────
# axis=1 means "column" (axis=0 would mean "row").
df_dropped = df.drop("Grade_Bonus", axis=1)
print(df_dropped)

# Note: drop() returns a NEW DataFrame by default — the original df is untouched
# unless you pass inplace=True, e.g. df.drop("Grade_Bonus", axis=1, inplace=True)

In [ ]:
# ─── Deleting a row ───────────────────────────────────────────────────────────
# Drop the row at index label 3 (axis=0 = row, which is the default so we can omit it)
df_dropped_row = df.drop(3)
print(df_dropped_row)

## 7. Handling missing data

Real-world data almost always has gaps. Pandas represents missing values as `NaN`
(Not a Number). Knowing how to find and handle them is essential.

In [ ]:
# ─── Create a DataFrame with some missing values on purpose ─────────────────
data_missing = {
    "Name":  ["Aman", "Priya", "Rohit", "Sneha"],
    "Marks": [85, np.nan, 78, np.nan],   # np.nan = pandas' "missing value" marker
    "City":  ["Delhi", "Mumbai", None, "Delhi"],
}
df_m = pd.DataFrame(data_missing)
print(df_m)

In [ ]:
# ─── Detecting missing values ────────────────────────────────────────────────
print("isna() — True where value is missing:")
print(df_m.isna())

print("\nCount of missing values per column:")
print(df_m.isna().sum())

In [ ]:
# ─── Dropping missing values ─────────────────────────────────────────────────
# dropna() removes any ROW that has AT LEAST ONE missing value, by default.
df_dropped_na = df_m.dropna()
print("After dropna():")
print(df_dropped_na)

In [ ]:
# ─── Filling missing values ──────────────────────────────────────────────────
# fillna() lets you replace NaN with something sensible instead of deleting rows.

# Fill numeric column with its mean (a common strategy)
df_filled = df_m.copy()   # .copy() so we don't accidentally modify df_m
df_filled["Marks"] = df_filled["Marks"].fillna(df_filled["Marks"].mean())

# Fill text column with a placeholder
df_filled["City"] = df_filled["City"].fillna("Unknown")

print(df_filled)

## 8. Sorting & ranking

In [ ]:
# ─── Sort by one column ──────────────────────────────────────────────────────
print("Sorted by Marks, ascending (default):")
print(df.sort_values("Marks"))

print("\nSorted by Marks, descending:")
print(df.sort_values("Marks", ascending=False))

In [ ]:
# ─── Sort by multiple columns ────────────────────────────────────────────────
# Sorts by City first, then breaks ties using Marks (descending).
print(df.sort_values(["City", "Marks"], ascending=[True, False]))

## 9. GroupBy — aggregation superpower

`groupby()` follows the classic **Split → Apply → Combine** pattern:
1. **Split** the data into groups (e.g. by City)
2. **Apply** a function to each group (e.g. mean)
3. **Combine** the results back into one table

This is the pandas equivalent of SQL's `GROUP BY`.

In [ ]:
# ─── Average marks per city ──────────────────────────────────────────────────
avg_marks_by_city = df.groupby("City")["Marks"].mean()
print(avg_marks_by_city)

In [ ]:
# ─── Multiple aggregations at once with .agg() ──────────────────────────────
summary = df.groupby("City")["Marks"].agg(["mean", "max", "min", "count"])
print(summary)

In [ ]:
# ─── Grouping by multiple columns ────────────────────────────────────────────
grouped = df.groupby(["City", "Passed"])["Marks"].mean()
print(grouped)

## 10. Merging, joining, and concatenating

Combine data from multiple DataFrames — very common when data is split across tables
(e.g. a "students" table and a separate "scores" table, like a SQL join).

In [ ]:
# ─── merge() — like a SQL JOIN, combines on a shared column ─────────────────
students = pd.DataFrame({
    "student_id": [1, 2, 3],
    "Name": ["Aman", "Priya", "Rohit"],
})

scores = pd.DataFrame({
    "student_id": [1, 2, 4],   # note: student_id 4 doesn't exist in `students`
    "Score": [85, 92, 60],
})

# how="inner" → only keep student_ids present in BOTH tables (default)
inner_merged = pd.merge(students, scores, on="student_id", how="inner")
print("Inner merge (only matching rows):")
print(inner_merged)

# how="left" → keep ALL rows from `students`, fill with NaN where no match
left_merged = pd.merge(students, scores, on="student_id", how="left")
print("\nLeft merge (all students, even without a score):")
print(left_merged)

In [ ]:
# ─── concat() — stack DataFrames on top of / beside each other ──────────────
batch1 = pd.DataFrame({"Name": ["Aman", "Priya"], "Marks": [85, 92]})
batch2 = pd.DataFrame({"Name": ["Rohit", "Sneha"], "Marks": [78, 88]})

# axis=0 (default) = stack ROWS on top of each other (like appending)
combined = pd.concat([batch1, batch2], ignore_index=True)  # ignore_index resets 0,1,2...
print(combined)

## 11. Pivot tables & reshaping

Pivot tables turn long, "row-per-record" data into a summarized cross-tab view —
just like a pivot table in Excel.

In [ ]:
# ─── pivot_table() — summarize data into a cross-tab ────────────────────────
sales = pd.DataFrame({
    "Region":  ["North", "North", "South", "South", "North", "South"],
    "Product": ["A", "B", "A", "B", "A", "B"],
    "Sales":   [100, 150, 200, 130, 120, 170],
})

# values = what to summarize, index = rows, columns = columns, aggfunc = how to summarize
pivot = sales.pivot_table(values="Sales", index="Region", columns="Product", aggfunc="sum")
print(pivot)

## 12. apply / map / lambda — custom transformations

When built-in pandas functions aren't enough, you can run your OWN function
on every value/row/column.

In [ ]:
# ─── Series.map() — transform every value in a Series ───────────────────────
# Great for simple value-by-value transformations (like a lookup or a small function).
df["Grade_Letter"] = df["Marks"].map(lambda m: "A" if m >= 85 else ("B" if m >= 70 else "C"))
print(df[["Name", "Marks", "Grade_Letter"]])

In [ ]:
# ─── DataFrame.apply() — run a function across an entire ROW ────────────────
# axis=1 means "apply this function to each ROW" (axis=0 would mean each column)
def summarize_row(row):
    return f"{row['Name']} scored {row['Marks']} in {row['City']}"

df["Summary"] = df.apply(summarize_row, axis=1)
print(df[["Name", "Summary"]])

## 13. String operations

Pandas gives every text column a `.str` accessor with vectorized string methods
(same names as Python's built-in string methods, but applied to the whole column at once).

In [ ]:
# ─── Common .str operations ──────────────────────────────────────────────────
print("Uppercase names:")
print(df["Name"].str.upper())

print("\nDoes name contain 'a' (case-insensitive)?")
print(df["Name"].str.contains("a", case=False))

print("\nLength of each name:")
print(df["Name"].str.len())

print("\nSplit City on nothing useful here, but this is how you'd split text:")
print(df["Name"].str.split("a"))   # splits each name on the letter 'a'

## 14. Date & time handling

Dates in raw files often arrive as plain TEXT (e.g. "2024-01-15").
`pd.to_datetime()` converts them into real datetime objects so you can do
date math, extract year/month/day, and filter by date ranges.

In [ ]:
# ─── Converting text to real datetime values ─────────────────────────────────
dates_df = pd.DataFrame({
    "Event": ["Signup", "First Purchase", "Renewal"],
    "Date":  ["2024-01-15", "2024-02-20", "2025-01-15"],
})

dates_df["Date"] = pd.to_datetime(dates_df["Date"])
print(dates_df)
print(dates_df.dtypes)   # Date column is now datetime64, not plain text

In [ ]:
# ─── Extracting parts of a date ──────────────────────────────────────────────
dates_df["Year"] = dates_df["Date"].dt.year
dates_df["Month"] = dates_df["Date"].dt.month
dates_df["Weekday"] = dates_df["Date"].dt.day_name()
print(dates_df)

In [ ]:
# ─── Date math ────────────────────────────────────────────────────────────────
# Subtracting two datetime columns gives you a Timedelta (a duration).
today = pd.Timestamp("2026-07-08")
dates_df["Days_Since_Event"] = (today - dates_df["Date"]).dt.days
print(dates_df[["Event", "Date", "Days_Since_Event"]])

## 15. Advanced topics

A quick tour of a few more powerful features you'll grow into as your projects get bigger.

In [ ]:
# ─── MultiIndex — grouping/indexing on MORE than one column at once ─────────
multi = df.groupby(["City", "Passed"])["Marks"].mean()
print("MultiIndex result:")
print(multi)
print("\nIndex levels:", multi.index.names)

# Use .reset_index() to turn a MultiIndex result back into a flat, normal DataFrame
flat = multi.reset_index()
print("\nFlattened back to a normal DataFrame:")
print(flat)

In [ ]:
# ─── Rolling / window functions — useful for time series (e.g. moving averages) ─
ts = pd.Series([10, 12, 13, 12, 15, 19, 21, 18], name="Daily_Sales")

# rolling(window=3) looks at each value + the 2 before it, then applies .mean()
ts_rolling_avg = ts.rolling(window=3).mean()
print("Original values:")
print(ts.tolist())
print("\n3-day rolling average (first 2 are NaN — not enough prior data yet):")
print(ts_rolling_avg.tolist())

In [ ]:
# ─── value_counts() — quick frequency count, extremely handy for exploring data ─
print(df["City"].value_counts())

In [ ]:
# ─── Performance tip: vectorize instead of looping ──────────────────────────
# BAD (slow): looping row by row in pure Python
import time

big_series = pd.Series(range(100_000))

start = time.time()
result_loop = [x * 2 for x in big_series]   # Python-level loop
loop_time = time.time() - start

start = time.time()
result_vectorized = big_series * 2          # pandas/numpy vectorized operation
vectorized_time = time.time() - start

print(f"Loop time      : {loop_time:.5f} sec")
print(f"Vectorized time: {vectorized_time:.5f} sec")
print("Vectorized is almost always faster — prefer it over manual loops.")

## 16. Finding & removing duplicate rows

Real data often has duplicate rows (e.g. the same form submitted twice).
Pandas makes it easy to spot and clean these up.

In [ ]:
# ─── Create a DataFrame with a duplicate row ─────────────────────────────────
dup_df = pd.DataFrame({
    "Name": ["Aman", "Priya", "Aman", "Rohit"],
    "Marks": [85, 92, 85, 78],
})
print(dup_df)

# duplicated() → True for rows that are exact repeats of an EARLIER row
print("\nWhich rows are duplicates?")
print(dup_df.duplicated())

# drop_duplicates() → keeps the FIRST occurrence, removes the rest
print("\nAfter drop_duplicates():")
print(dup_df.drop_duplicates())

# subset= lets you check duplicates based on only SOME columns
print("\nDuplicates based on 'Name' only:")
print(dup_df.drop_duplicates(subset=["Name"]))

## 17. set_index() and reset_index()

By default, rows are labeled 0, 1, 2... You can promote a meaningful column
(like an ID) to be the index instead — this makes `.loc[]` lookups by that ID very fast.

In [ ]:
# ─── set_index() — make a column the new row index ──────────────────────────
indexed_df = df.set_index("Name")
print(indexed_df)

# Now you can look up a row directly by name using .loc[]
print("\nLookup by name label:")
print(indexed_df.loc["Priya"])

In [ ]:
# ─── reset_index() — undo set_index(), go back to plain 0,1,2... numbering ──
back_to_normal = indexed_df.reset_index()
print(back_to_normal)

## 18. Iterating over rows — and why you usually shouldn't

Sometimes you genuinely need to loop row by row. Pandas gives you tools for it —
but remember: **vectorized operations (Section 15) are almost always faster.**
Only loop when there's no vectorized way to express the logic.

In [ ]:
# ─── iterrows() — loop over (index, row) pairs ──────────────────────────────
# Returns each row as a Series — convenient, but SLOW on large data.
for idx, row in df.head(2).iterrows():
    print(f"Row {idx}: {row['Name']} scored {row['Marks']}")

# ─── itertuples() — faster alternative, returns lightweight namedtuples ─────
print()
for row in df.head(2).itertuples():
    print(f"{row.Name} scored {row.Marks}")

## 19. Reshaping: melt(), stack(), unstack()

`pivot_table()` (Section 11) makes wide tables. `melt()` does the opposite —
it turns WIDE data into LONG data. This "long format" is often what plotting
libraries and statistical tools expect.

In [ ]:
# ─── melt() — wide format → long format ──────────────────────────────────────
wide = pd.DataFrame({
    "Name": ["Aman", "Priya"],
    "Math": [85, 92],
    "Science": [78, 88],
})
print("Wide format:")
print(wide)

# id_vars = column(s) to keep as-is; the rest get "melted" into Subject/Score pairs
long = wide.melt(id_vars="Name", var_name="Subject", value_name="Score")
print("\nLong format (after melt):")
print(long)

In [ ]:
# ─── stack() / unstack() — pivot between column-levels and row-levels ───────
# stack(): move the innermost COLUMN level into the ROW index
stacked = wide.set_index("Name").stack()
print("Stacked (columns became part of the row index):")
print(stacked)

# unstack(): the reverse — move a ROW index level back into columns
print("\nUnstacked (back to wide format):")
print(stacked.unstack())

## 20. Categorical data type

When a text column only has a small, fixed set of repeating values
(e.g. "Small"/"Medium"/"Large"), converting it to `category` dtype saves memory
and lets you define a meaningful ORDER for sorting/comparison.

In [ ]:
# ─── Converting a column to category dtype ───────────────────────────────────
sizes = pd.DataFrame({"Item": ["Shirt", "Mug", "Jacket"], "Size": ["M", "S", "L"]})
sizes["Size"] = sizes["Size"].astype("category")
print(sizes.dtypes)

# ─── Defining a custom ORDER for the categories ──────────────────────────────
size_order = pd.CategoricalDtype(categories=["S", "M", "L"], ordered=True)
sizes["Size"] = sizes["Size"].astype(size_order)

# Now sorting respects S < M < L, not alphabetical order
print("\nSorted by the meaningful size order:")
print(sizes.sort_values("Size"))

## 21. query() and eval() — readable filtering with SQL-like syntax

`df.query()` is an alternative to the boolean-indexing style from Section 5.
Many people find it more readable, especially with several conditions.

In [ ]:
# ─── query() — filter rows using a SQL-like string expression ───────────────
result = df.query("Marks > 80 and City == 'Delhi'")
print(result)

# You can also reference a Python variable inside query() with an @ prefix
min_marks = 80
result2 = df.query("Marks > @min_marks")
print("\nUsing a variable in query():")
print(result2)

## 22. Exporting data to other formats

Just as pandas can READ many formats (Section 4), it can WRITE them too.

In [ ]:
# ─── Common export methods ───────────────────────────────────────────────────
df.to_csv("export_demo.csv", index=False)     # CSV file
df.to_json("export_demo.json", orient="records")  # JSON file
# df.to_excel("export_demo.xlsx", index=False)     # needs openpyxl installed

# ─── Converting to plain Python structures ───────────────────────────────────
print("As a list of dicts (records):")
print(df[["Name", "Marks"]].to_dict(orient="records"))

print("\nAs a plain nested dict:")
print(df[["Name", "Marks"]].to_dict())

## 23. Reading large files in chunks

If a CSV is too big to fit comfortably in memory, `chunksize=` lets you process
it piece by piece instead of loading it all at once.

In [ ]:
# ─── Process a file in chunks instead of all at once ────────────────────────
# chunksize=2 here just for demonstration (in real use it'd be something like 100_000)
total_rows = 0
total_marks = 0

for chunk in pd.read_csv("sample_students.csv", chunksize=2):
    total_rows += len(chunk)
    total_marks += chunk["Marks"].sum()
    print(f"Processed a chunk of {len(chunk)} rows")

print(f"\nTotal rows processed: {total_rows}")
print(f"Total marks summed  : {total_marks}")

## 24. Quick plotting straight from pandas

Pandas has `.plot()` built in (it uses matplotlib under the hood) — handy for
quick, exploratory charts without importing matplotlib separately.

In [ ]:
# ─── Requires matplotlib to be installed: pip install matplotlib ────────────
import matplotlib.pyplot as plt

df.plot(x="Name", y="Marks", kind="bar", legend=False, title="Marks by Student")
plt.ylabel("Marks")
plt.tight_layout()
plt.show()

## 🎉 Summary

You've now covered pandas from basic to advanced:
- Series & DataFrame basics
- Reading, inspecting, selecting, and filtering data
- Adding/editing/deleting columns & rows
- Handling missing data
- Sorting, grouping, merging, pivoting
- Custom transformations with apply/map
- String and datetime operations
- MultiIndex, rolling windows, and performance tips
- Duplicates, custom indexes, and row iteration
- Reshaping with melt/stack/unstack
- Categorical dtype, query()/eval(), exporting, chunked reading, and quick plots

**Next steps:** try loading a real CSV file with `pd.read_csv("your_file.csv")`
and practice these same operations on it — that's the fastest way to make this stick.